## Importing Libraries

In [ ]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
!pip install simpletransformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.8/250.8 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 33.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 42.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 69.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 74.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 36.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 25.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.6/248.6 kB 22.7 MB/s eta

In [ ]:
!pip install transformers

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
!pip install textblob

In [ ]:
from textblob import TextBlob

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer as sia
from tqdm.notebook import tqdm

## **Dataset Cleaning**


Reading CSV File

In [ ]:
df = pd.read_csv("/content/raw_data.csv")

In [ ]:
df.head()

,Sno,Date,Headline,Covid,Sentiment,Description,Image,Source
0,0,2020-04-29,"Coronavirus cases reach 2,438 in Rajasthan; 81...",1,0,Rajasthan on Wednesday reported 74 new coronav...,https://static.inshorts.com/inshorts/images/v1...,http://www.rajswasthya.nic.in/?utm_campaign=fu...
1,1,2020-04-29,"Coronavirus cases in Delhi surge to 3,439 afte...",1,0,The total number of coronavirus cases in Delhi...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/shemin_joy/status/12555489...
2,2,2020-04-30,"Anguished, I'll always recall our interactions...",0,1,"Condoling the demise of actor Rishi Kapoor, PM...",https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/narendramodi/status/125573...
3,3,2020-04-30,It's a terrible week for Indian cinema: Rahul ...,0,0,Congress leader Rahul Gandhi condoled Rishi Ka...,https://static.inshorts.com/inshorts/images/v1...,https://zeenews.india.com/india/politicians-mo...
4,4,2020-04-30,Delhi Police perform 'parikrama' of AIIMS to t...,1,1,As a sign of respect for healthcare profession...,https://static.inshorts.com/inshorts/images/v1...,https://zeenews.india.com/india/delhi-police-p...


# Checking for null and duplicates


no nulls and duplicates

In [ ]:
df.isnull().sum()

Sno            0
Date           0
Headline       0
Covid          0
Sentiment      0
Description    0
Image          0
Source         0
dtype: int64

In [ ]:
df.duplicated().sum()

0

# Removing columns that are not needed

In [ ]:
df.columns

Index(['Sno', 'Date', 'Headline', 'Covid', 'Sentiment', 'Description', 'Image',
       'Source'],
      dtype='object')

In [ ]:
df = df.drop(["Sno", "Covid", "Description", "Image", "Source"], axis = 1)

In [ ]:
df.shape

(4072, 3)

In [ ]:
df.columns

Index(['Date', 'Headline', 'Sentiment'], dtype='object')

cleaned dataset


## VADER model

In [ ]:
sid = sia()

few examples

In [ ]:
sid.polarity_scores("i am happy!")

{'neg': 0.0, 'neu': 0.2, 'pos': 0.8, 'compound': 0.6114}

In [ ]:
sid.polarity_scores("i am devastated")

{'neg': 0.8, 'neu': 0.2, 'pos': 0.0, 'compound': -0.6124}

 Polarity scores for dataset

In [ ]:
#Run the polarity scores on the entire dataset
res = {}
for i, row in tqdm(df.iterrows(), total = len(df)):
  text = row['Headline']
  date = row['Date']
  res[date] = sid.polarity_scores(text)

  0%|          | 0/4072 [00:00<?, ?it/s]

In [ ]:
vaders = pd.DataFrame(res).T
vaders = vaders.reset_index().rename(columns = {'index' : 'Date'})
vaders = vaders.merge(df, how = 'left')

In [ ]:
vaders.head()

,Date,neg,neu,pos,compound,Sno,Headline,Covid,Sentiment,Description,Image,Source
0,2020-04-29,0.174,0.826,0.0,-0.2732,0,"Coronavirus cases reach 2,438 in Rajasthan; 81...",1,0,Rajasthan on Wednesday reported 74 new coronav...,https://static.inshorts.com/inshorts/images/v1...,http://www.rajswasthya.nic.in/?utm_campaign=fu...
1,2020-04-29,0.174,0.826,0.0,-0.2732,1,"Coronavirus cases in Delhi surge to 3,439 afte...",1,0,The total number of coronavirus cases in Delhi...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/shemin_joy/status/12555489...
2,2020-04-29,0.174,0.826,0.0,-0.2732,19,"Man burnt alive, 7 houses and temple gutted in...",0,0,A 35-year-old man was burnt alive while two pe...,https://static.inshorts.com/inshorts/images/v1...,https://covid-positive-news.herokuapp.com
3,2020-04-29,0.174,0.826,0.0,-0.2732,24,81 coronavirus patients recover in Gautam Budd...,1,1,Gautam Buddha Nagar District Magistrate Suhas ...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/dmgbnagar/status/125545866...
4,2020-04-29,0.174,0.826,0.0,-0.2732,25,Colleges to reopen for old students from Aug 1...,1,1,The UGC today said that the 2020-21 academic s...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/TimesNow/status/1255507208...


In [ ]:
vaders['compound']

0      -0.2732
1      -0.2732
2      -0.2732
3      -0.2732
4      -0.2732
         ...  
4067   -0.2263
4068   -0.2263
4069   -0.2263
4070   -0.2263
4071   -0.2263
Name: compound, Length: 4072, dtype: float64

In [ ]:
vaders['Sentiment-score'] = vaders['Sentiment'].apply(lambda c : 'pos' if c > 0 else 'neg')
vaders.head()

,Date,neg,neu,pos,compound,Sno,Headline,Covid,Sentiment,Description,Image,Source,Sentiment-score
0,2020-04-29,0.174,0.826,0.0,-0.2732,0,"Coronavirus cases reach 2,438 in Rajasthan; 81...",1,0,Rajasthan on Wednesday reported 74 new coronav...,https://static.inshorts.com/inshorts/images/v1...,http://www.rajswasthya.nic.in/?utm_campaign=fu...,neg
1,2020-04-29,0.174,0.826,0.0,-0.2732,1,"Coronavirus cases in Delhi surge to 3,439 afte...",1,0,The total number of coronavirus cases in Delhi...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/shemin_joy/status/12555489...,neg
2,2020-04-29,0.174,0.826,0.0,-0.2732,19,"Man burnt alive, 7 houses and temple gutted in...",0,0,A 35-year-old man was burnt alive while two pe...,https://static.inshorts.com/inshorts/images/v1...,https://covid-positive-news.herokuapp.com,neg
3,2020-04-29,0.174,0.826,0.0,-0.2732,24,81 coronavirus patients recover in Gautam Budd...,1,1,Gautam Buddha Nagar District Magistrate Suhas ...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/dmgbnagar/status/125545866...,pos
4,2020-04-29,0.174,0.826,0.0,-0.2732,25,Colleges to reopen for old students from Aug 1...,1,1,The UGC today said that the 2020-21 academic s...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/TimesNow/status/1255507208...,pos


In [ ]:
vaders['Compound-score'] = vaders['compound'].apply(lambda c : 'pos' if c > 0 else 'neg')
vaders.head()

,Date,neg,neu,pos,compound,Sno,Headline,Covid,Sentiment,Description,Image,Source,Sentiment-score,Compound-score
0,2020-04-29,0.174,0.826,0.0,-0.2732,0,"Coronavirus cases reach 2,438 in Rajasthan; 81...",1,0,Rajasthan on Wednesday reported 74 new coronav...,https://static.inshorts.com/inshorts/images/v1...,http://www.rajswasthya.nic.in/?utm_campaign=fu...,neg,neg
1,2020-04-29,0.174,0.826,0.0,-0.2732,1,"Coronavirus cases in Delhi surge to 3,439 afte...",1,0,The total number of coronavirus cases in Delhi...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/shemin_joy/status/12555489...,neg,neg
2,2020-04-29,0.174,0.826,0.0,-0.2732,19,"Man burnt alive, 7 houses and temple gutted in...",0,0,A 35-year-old man was burnt alive while two pe...,https://static.inshorts.com/inshorts/images/v1...,https://covid-positive-news.herokuapp.com,neg,neg
3,2020-04-29,0.174,0.826,0.0,-0.2732,24,81 coronavirus patients recover in Gautam Budd...,1,1,Gautam Buddha Nagar District Magistrate Suhas ...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/dmgbnagar/status/125545866...,pos,neg
4,2020-04-29,0.174,0.826,0.0,-0.2732,25,Colleges to reopen for old students from Aug 1...,1,1,The UGC today said that the 2020-21 academic s...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/TimesNow/status/1255507208...,pos,neg


In [ ]:
print(confusion_matrix(vaders['Sentiment-score'], vaders['Compound-score']))

[[1897  845]
 [ 989  341]]


In [ ]:
# VADER Result
print(classification_report(vaders['Sentiment-score'], vaders['Compound-score']))

              precision    recall  f1-score   support

         neg       0.66      0.69      0.67      2742
         pos       0.29      0.26      0.27      1330

    accuracy                           0.55      4072
   macro avg       0.47      0.47      0.47      4072
weighted avg       0.54      0.55      0.54      4072



In [ ]:
accuracy_score(vaders['Sentiment-score'], vaders['Compound-score'])

0.5496070726915521

## Roberta Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train, test = train_test_split(df, test_size = 0.2)


In [ ]:
from simpletransformers.classification import ClassificationModel

model = ClassificationModel('roberta', f"cardiffnlp/twitter-roberta-base-sentiment", num_labels = 3, args =
 {
    'reprocess_input_data' : True,
    'overwrite_output_dir' : True
  },
                            use_cuda = False)

(…)-base-sentiment/resolve/main/config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

(…)a-base-sentiment/resolve/main/vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

(…)a-base-sentiment/resolve/main/merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

(…)ent/resolve/main/special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
trainr_df = pd.DataFrame({'text' : train['Headline'].replace(r'\n', ' ', regex = True), 'label' : train['Sentiment']})
evalr_df = pd.DataFrame({'text' : test['Headline'].replace(r'\n', ' ', regex = True), 'label' : test['Sentiment']})


In [ ]:
model.train_model(trainr_df)

/usr/local/lib/python3.10/dist-packages/simpletransformers/classification/classification_model.py:612: UserWarning: Dataframe headers not specified. Falling back to using column 0 as text and column 1 as labels.
  warnings.warn(


  0%|          | 0/3257 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Running Epoch 0 of 1:   0%|          | 0/408 [00:00<?, ?it/s]

(408, 0.45494529815828977)

In [ ]:
result, model_outputs, wrong_predictions = model.eval_model(evalr_df)


/usr/local/lib/python3.10/dist-packages/simpletransformers/classification/classification_model.py:1454: UserWarning: Dataframe headers not specified. Falling back to using column 0 as text and column 1 as labels.
  warnings.warn(


  0%|          | 0/815 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
result


{'mcc': 0.7302354577304353, 'eval_loss': 0.3317710743559634}

In [ ]:
model_outputs


array([[ 1.42330563,  2.71583414, -4.74672127],
       [-0.55231309,  3.34715343, -3.09735751],
       [ 3.49487925,  1.12124646, -4.98806095],
       ...,
       [-0.48252428,  3.44703436, -3.31117988],
       [ 1.49853683,  2.64936495, -4.7543745 ],
       [ 1.41854262,  2.69802332, -4.59067249]])

In [ ]:
lst = []
for arr in model_outputs:
  lst.append(np.argmax(arr))


In [ ]:
true = evalr_df['label'].tolist()
predicted = lst


In [ ]:
import sklearn
mat = sklearn.metrics.confusion_matrix(true, predicted)
mat

array([[460,  69],
       [ 35, 251]])

In [ ]:
#Roberta Result
sklearn.metrics.accuracy_score(true, predicted)

0.8723926380368098

In [ ]:
sklearn.metrics.f1_score(true, predicted)


0.8283828382838283

## BERT Model

In [ ]:
# Test and Train the data
train, test = train_test_split(df, test_size = 0.2)

Building a model

In [ ]:
from simpletransformers.classification import ClassificationModel

model = ClassificationModel('bert', 'bert-base-cased', num_labels = 3, args =
 {
    'reprocess_input_data' : True,
    'overwrite_output_dir' : True
  },
                            use_cuda = False)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainb_df = pd.DataFrame({'text' : train['Headline'].replace(r'\n', ' ', regex = True), 'label' : train['Sentiment']})
evalb_df = pd.DataFrame({'text' : test['Headline'].replace(r'\n', ' ', regex = True), 'label' : test['Sentiment']})

In [ ]:
model.train_model(trainb_df)

  0%|          | 0/3257 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Running Epoch 0 of 1:   0%|          | 0/408 [00:00<?, ?it/s]

(408, 0.45020649556810144)

In [ ]:
result, model_outputs, wrong_predictions = model.eval_model(evalb_df)

  0%|          | 0/815 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
result

{'mcc': 0.6382488048800944, 'eval_loss': 0.4186269024606137}

In [ ]:
model_outputs

array([[ 3.86278844, -0.84099877, -2.96068192],
       [ 3.1927228 ,  0.41430053, -3.17711639],
       [ 3.03764844,  1.0657742 , -3.58570981],
       ...,
       [ 3.45069766,  0.40950403, -3.55495739],
       [ 3.99971533, -0.81417567, -3.0009048 ],
       [ 1.43140972,  2.16726494, -3.06155849]])

In [ ]:
lst = []
for arr in model_outputs:
  lst.append(np.argmax(arr))

In [ ]:
true = evalb_df['label'].tolist()
predicted = lst

In [ ]:
import sklearn
mat = sklearn.metrics.confusion_matrix(true, predicted)
mat

array([[473,  82],
       [ 51, 209]])

In [ ]:
sklearn.metrics.classification_report(true, predicted, target_names = ['positive', 'negative'])

'              precision    recall  f1-score   support\n\n    positive       0.90      0.85      0.88       555\n    negative       0.72      0.80      0.76       260\n\n    accuracy                           0.84       815\n   macro avg       0.81      0.83      0.82       815\nweighted avg       0.84      0.84      0.84       815\n'

In [ ]:
#BERT Result
sklearn.metrics.accuracy_score(true, predicted)

0.8368098159509203

In [ ]:
sklearn.metrics.f1_score(true, predicted)

0.7586206896551725

## TextBlob Model

In [ ]:
df['polarity_score'] = df['Headline'].apply(lambda s: TextBlob(s).sentiment.polarity)
df.head()


,Sno,Date,Headline,Covid,Sentiment,Description,Image,Source,polarity_score
0,0,2020-04-29,"Coronavirus cases reach 2,438 in Rajasthan; 81...",1,0,Rajasthan on Wednesday reported 74 new coronav...,https://static.inshorts.com/inshorts/images/v1...,http://www.rajswasthya.nic.in/?utm_campaign=fu...,0.000000
1,1,2020-04-29,"Coronavirus cases in Delhi surge to 3,439 afte...",1,0,The total number of coronavirus cases in Delhi...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/shemin_joy/status/12555489...,0.136364
2,2,2020-04-30,"Anguished, I'll always recall our interactions...",0,1,"Condoling the demise of actor Rishi Kapoor, PM...",https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/narendramodi/status/125573...,0.000000
3,3,2020-04-30,It's a terrible week for Indian cinema: Rahul ...,0,0,Congress leader Rahul Gandhi condoled Rishi Ka...,https://static.inshorts.com/inshorts/images/v1...,https://zeenews.india.com/india/politicians-mo...,-1.000000
4,4,2020-04-30,Delhi Police perform 'parikrama' of AIIMS to t...,1,1,As a sign of respect for healthcare profession...,https://static.inshorts.com/inshorts/images/v1...,https://zeenews.india.com/india/delhi-police-p...,0.000000


In [ ]:
import sklearn

In [ ]:
df['Sentiment-score'] = df['Sentiment'].apply(lambda c : 'pos' if c > 0 else 'neg')
df.head()

,Sno,Date,Headline,Covid,Sentiment,Description,Image,Source,polarity_score,Sentiment-score
0,0,2020-04-29,"Coronavirus cases reach 2,438 in Rajasthan; 81...",1,0,Rajasthan on Wednesday reported 74 new coronav...,https://static.inshorts.com/inshorts/images/v1...,http://www.rajswasthya.nic.in/?utm_campaign=fu...,0.000000,neg
1,1,2020-04-29,"Coronavirus cases in Delhi surge to 3,439 afte...",1,0,The total number of coronavirus cases in Delhi...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/shemin_joy/status/12555489...,0.136364,neg
2,2,2020-04-30,"Anguished, I'll always recall our interactions...",0,1,"Condoling the demise of actor Rishi Kapoor, PM...",https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/narendramodi/status/125573...,0.000000,pos
3,3,2020-04-30,It's a terrible week for Indian cinema: Rahul ...,0,0,Congress leader Rahul Gandhi condoled Rishi Ka...,https://static.inshorts.com/inshorts/images/v1...,https://zeenews.india.com/india/politicians-mo...,-1.000000,neg
4,4,2020-04-30,Delhi Police perform 'parikrama' of AIIMS to t...,1,1,As a sign of respect for healthcare profession...,https://static.inshorts.com/inshorts/images/v1...,https://zeenews.india.com/india/delhi-police-p...,0.000000,pos


In [ ]:
df['polarity-score'] = df['polarity_score'].apply(lambda c : 'pos' if c > 0 else 'neg')
df.head()

,Sno,Date,Headline,Covid,Sentiment,Description,Image,Source,polarity_score,Sentiment-score,polarity-score
0,0,2020-04-29,"Coronavirus cases reach 2,438 in Rajasthan; 81...",1,0,Rajasthan on Wednesday reported 74 new coronav...,https://static.inshorts.com/inshorts/images/v1...,http://www.rajswasthya.nic.in/?utm_campaign=fu...,0.000000,neg,neg
1,1,2020-04-29,"Coronavirus cases in Delhi surge to 3,439 afte...",1,0,The total number of coronavirus cases in Delhi...,https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/shemin_joy/status/12555489...,0.136364,neg,pos
2,2,2020-04-30,"Anguished, I'll always recall our interactions...",0,1,"Condoling the demise of actor Rishi Kapoor, PM...",https://static.inshorts.com/inshorts/images/v1...,https://twitter.com/narendramodi/status/125573...,0.000000,pos,neg
3,3,2020-04-30,It's a terrible week for Indian cinema: Rahul ...,0,0,Congress leader Rahul Gandhi condoled Rishi Ka...,https://static.inshorts.com/inshorts/images/v1...,https://zeenews.india.com/india/politicians-mo...,-1.000000,neg,neg
4,4,2020-04-30,Delhi Police perform 'parikrama' of AIIMS to t...,1,1,As a sign of respect for healthcare profession...,https://static.inshorts.com/inshorts/images/v1...,https://zeenews.india.com/india/delhi-police-p...,0.000000,pos,neg


In [ ]:
print(confusion_matrix(df['Sentiment-score'], df['polarity-score']))

[[2169  573]
 [1041  289]]


In [ ]:
#TextBlob Result
print(classification_report(df['Sentiment-score'], df['polarity-score']))

              precision    recall  f1-score   support

         neg       0.68      0.79      0.73      2742
         pos       0.34      0.22      0.26      1330

    accuracy                           0.60      4072
   macro avg       0.51      0.50      0.50      4072
weighted avg       0.56      0.60      0.58      4072



In [ ]:
accuracy_score(df['Sentiment-score'], df['polarity-score'])

0.6036345776031434